# Lab 06-05 · Mesh Analysis, Loop Analysis and Shared Returns

Companion to chapter [06-05](../../chapters/06-05-mesh-analysis.md). Handout: [README.md](README.md).

Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import shutil
import subprocess
from collections import deque
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HAVE_NGSPICE = shutil.which("ngspice") is not None
print("ngspice available:", HAVE_NGSPICE)

ngspice available: True


## Part 1 — Mesh matrices from a drawing

Describe each branch by its kind, its value and the meshes that pass through it, each with a sign: +1 if the mesh traverses the branch in the branch's reference direction, −1 if against it. The solver below builds $\mathbf{R} = \sum_b R_b\,\mathbf{m}_b\mathbf{m}_b^\mathsf{T}$ and the source vector, and handles current sources by adding each one's voltage as an extra unknown with a constraint row: the exact dual of modified nodal analysis (06-04).

Branch kinds: `("R", ohms, {mesh: sign})`, `("V", volts, {mesh: sign})` with the rise in the branch's reference direction, and `("I", amps, {mesh: sign})` with the current in the branch's reference direction.

**🔮 Predict:** for the bridge of Example 06-05.2, which entry of $\mathbf{R}$ is largest, and is the matrix symmetric?

In [2]:
def mesh_solve(branches, n_mesh):
    '''Modified mesh analysis. Returns (mesh currents, current-source voltages (rises), R, v).'''
    m_vec = lambda signs: np.array([signs.get(k, 0) for k in range(n_mesh)], dtype=float)
    R = np.zeros((n_mesh, n_mesh)); v = np.zeros(n_mesh); isrc = []
    for kind, val, signs in branches:
        m = m_vec(signs)
        if kind == "R":
            R += val * np.outer(m, m)
        elif kind == "V":
            v += val * m
        elif kind == "I":
            isrc.append((val, m))
    k = len(isrc)
    A = np.zeros((n_mesh + k, n_mesh + k)); z = np.zeros(n_mesh + k)
    A[:n_mesh, :n_mesh] = R; z[:n_mesh] = v
    for j, (val, m) in enumerate(isrc):
        A[:n_mesh, n_mesh + j] = -m           # the source's unknown voltage acts like a source rise
        A[n_mesh + j, :n_mesh] = m            # constraint: branch current m.i equals the source current
        z[n_mesh + j] = val
    x = np.linalg.solve(A, z)
    return x[:n_mesh], x[n_mesh:], R, v

# Example 06-05.1: meshes 0 and 1 clockwise. Branch reference directions: along mesh 0 for its own
# branches and for the shared R2 (down); along mesh 1 for its own branches.
ex1 = [("V", 20, {0: 1}), ("R", 2, {0: 1}), ("R", 4, {0: 1, 1: -1}),
       ("R", 4, {1: 1}), ("V", -8, {1: 1})]         # the 8 V battery is a drop in mesh 1's direction
i, _, R, v = mesh_solve(ex1, 2)
print("Example 06-05.1: R =", R.tolist(), "v =", v.tolist(), "-> i =", np.round(i, 6))
assert np.allclose(i, [4, 1])

# Example 06-05.2: mesh 0 = source loop, 1 = upper window, 2 = lower window
ex2 = [("V", 10, {0: 1}), ("R", 10, {0: 1, 1: -1}), ("R", 40, {0: 1, 2: -1}),
       ("R", 20, {1: 1}), ("R", 30, {1: 1, 2: -1}), ("R", 50, {2: 1})]
i2, _, R2, _ = mesh_solve(ex2, 3)
print("\nExample 06-05.2: R =\n", R2, "\n-> i =", np.round(i2, 6), " bridge current c->d =", round(1e3 * (i2[2] - i2[1]), 3), "mA")
assert np.allclose(R2, R2.T) and abs(i2[0] - 0.344262) < 1e-6

# Example 06-05.3: the 3 A source in the shared branch, reference direction upward (= along mesh 1)
ex3 = [("V", 36, {0: 1}), ("R", 4, {0: 1}), ("I", 3, {0: -1, 1: 1}), ("R", 2, {1: 1}), ("R", 6, {1: 1})]
i3, u3, _, _ = mesh_solve(ex3, 2)
print("\nExample 06-05.3: i =", np.round(i3, 6), " current-source voltage (rise along its arrow) =", np.round(u3, 6))
assert np.allclose(i3, [1, 4]) and np.allclose(u3, [32])

Example 06-05.1: R = [[6.0, -4.0], [-4.0, 8.0]] v = [20.0, -8.0] -> i = [4. 1.]

Example 06-05.2: R =
 [[ 50. -10. -40.]
 [-10.  60. -30.]
 [-40. -30. 120.]] 
-> i = [0.344262 0.131148 0.147541]  bridge current c->d = 16.393 mA

Example 06-05.3: i = [1. 4.]  current-source voltage (rise along its arrow) = [32.]


The largest entry is $R_{33} = 120$ Ω, the total resistance around the lower window, and the matrix is symmetric. In Example 06-05.3 the solver never formed a supermesh explicitly: the current source's unknown voltage appears in both meshes' KVL rows and the constraint row fixes the difference of their currents. Eliminating that voltage by adding the two KVL rows *is* the supermesh equation.

## Part 2 — Mesh against nodal against ngspice

**🔮 Predict:** for Examples 06-05.1 to 06-05.3, how many unknowns does each method need once the sources are used?

In [3]:
def nodal_solve(resistors, vsrcs=(), isrcs=()):
    '''Plain nodal analysis with grounded voltage sources only: resistors (a, b, ohms);
    vsrcs (node, volts) fix nodes; isrcs (node, amps) push current into a node.'''
    fixed = dict(vsrcs)
    nodes = sorted({n for a, b, _ in resistors for n in (a, b)} - {"0"} - set(fixed))
    idx = {n: k for k, n in enumerate(nodes)}
    G = np.zeros((len(nodes), len(nodes))); i = np.zeros(len(nodes))
    for a, b, r in resistors:
        for p, q in ((a, b), (b, a)):
            if p in idx:
                G[idx[p], idx[p]] += 1 / r
                if q in idx:
                    G[idx[p], idx[q]] -= 1 / r
                elif q in fixed:
                    i[idx[p]] += fixed[q] / r
    for n, amps in isrcs:
        i[idx[n]] += amps
    vals = np.linalg.solve(G, i) if nodes else np.array([])
    return {**fixed, "0": 0.0, **dict(zip(nodes, vals))}, len(nodes)

n1, k1 = nodal_solve([("a", "m", 2), ("m", "0", 4), ("m", "c", 4)], vsrcs=[("a", 20), ("c", 8)])
n2, k2 = nodal_solve([("a", "c", 10), ("a", "d", 20), ("c", "d", 30), ("c", "0", 40), ("d", "0", 50)], vsrcs=[("a", 10)])
n3, k3 = nodal_solve([("a", "t", 4), ("t", "r", 2), ("r", "0", 6)], vsrcs=[("a", 36)], isrcs=[("t", 3)])
print(f"Example 1: mesh 2 unknowns, nodal {k1}; v_m = {n1['m']:.6f} V -> i1 = {(20 - n1['m']) / 2:.6f} A")
print(f"Example 2: mesh 3 unknowns, nodal {k2}; source current = {(10 - n2['c']) / 10 + (10 - n2['d']) / 20:.6f} A")
print(f"Example 3: mesh 1 unknown (2 minus the shared source), nodal {k3}; i2 = {n3['r'] / 6:.6f} A")
assert (k1, k2, k3) == (1, 2, 2)

if HAVE_NGSPICE:
    out = subprocess.run(["ngspice", "-b", "meshes.cir"], capture_output=True, text=True).stdout
    ng = {}
    for line in out.splitlines():
        parts = line.split("=")
        if len(parts) == 2:
            try:
                ng[parts[0].strip()] = float(parts[1])
            except ValueError:
                pass
    pairs = [("ex1i1", i[0]), ("ex1i2", i[1]), ("ex2i1", -i2[0]), ("ex2icd", i2[2] - i2[1]), ("ex3i2", i3[1]), ("ex3vcs", u3[0])]
    for name, mine in pairs:
        print(f"{name:7s} mesh {mine: .6g}   ngspice {ng[name]: .6g}")
        assert abs(mine - ng[name]) <= 1e-5 * max(1.0, abs(mine))
else:
    print("ngspice not found; see SETUP.md. Skipping the comparison.")

Example 1: mesh 2 unknowns, nodal 1; v_m = 12.000000 V -> i1 = 4.000000 A
Example 2: mesh 3 unknowns, nodal 2; source current = 0.344262 A
Example 3: mesh 1 unknown (2 minus the shared source), nodal 2; i2 = 4.000000 A
ex1i1   mesh  4   ngspice  4
ex1i2   mesh  1   ngspice  1
ex2i1   mesh -0.344262   ngspice -0.344262
ex2icd  mesh  0.0163934   ngspice  0.0163934
ex3i2   mesh  4   ngspice  4
ex3vcs  mesh  32   ngspice  32


## Part 3 — Loop analysis of a nonplanar circuit

$K_{3,3}$ (Figure 2 of the chapter) has no meshes. The code below finds a spanning tree by breadth-first search, forms one fundamental loop per link, builds the loop matrix $\mathbf{B}$ (loops × branches) and solves $\mathbf{B}\,\mathrm{diag}(R)\,\mathbf{B}^\mathsf{T}\,\mathbf{i}_{loop} = \mathbf{B}\,\mathbf{e}$, where $\mathbf{e}$ lists each branch's source rise in its reference direction. Every branch is referenced from its first node to its second.

In [4]:
def loop_analysis(branches, root):
    '''branches: list of (a, b, R, E): resistance R and source rise E met when going from a to b
    through the branch. Returns loop currents, branch currents (a -> b), the loop matrix B and the tree.'''
    nodes = sorted({n for a, b, _, _ in branches for n in (a, b)})
    adj = {n: [] for n in nodes}
    for k, (a, b, _, _) in enumerate(branches):
        adj[a].append((b, k)); adj[b].append((a, k))
    parent = {root: None}; queue = deque([root])
    while queue:                                   # breadth-first spanning tree
        n = queue.popleft()
        for m, k in adj[n]:
            if m not in parent:
                parent[m] = (n, k); queue.append(m)
    tree = {pk[1] for pk in parent.values() if pk}
    links = [k for k in range(len(branches)) if k not in tree]

    def path_to_root(n):                           # list of (branch, from, to) walking up the tree
        steps = []
        while parent[n]:
            p, k = parent[n]; steps.append((k, n, p)); n = p
        return steps

    B = np.zeros((len(links), len(branches)))
    for row, k in enumerate(links):
        a, b, _, _ = branches[k]
        B[row, k] = 1.0                            # traverse the link from a to b ...
        up_b, up_a = path_to_root(b), path_to_root(a)
        for kk, frm, to in up_b:                   # ... then from b up to the root ...
            B[row, kk] += 1.0 if branches[kk][0] == frm else -1.0
        for kk, frm, to in up_a:                   # ... then down from the root to a
            B[row, kk] -= 1.0 if branches[kk][0] == frm else -1.0
    Rb = np.array([r for _, _, r, _ in branches]); Eb = np.array([e for _, _, _, e in branches])
    i_loop = np.linalg.solve(B @ np.diag(Rb) @ B.T, B @ Eb)
    return i_loop, B.T @ i_loop, B, tree

k33 = [("1", "4", 1.0, -10.0)]                     # going 1 -> 4 crosses the source from + to -: a 10 V drop
k33 += [(str(a), str(b), 1.0, 0.0) for a in (1, 2, 3) for b in (4, 5, 6) if (a, b) != (1, 4)]
i_loop, i_branch, B, tree = loop_analysis(k33, root="1")
print("tree branches:", sorted(f"{k33[k][0]}-{k33[k][1]}" for k in tree))
print("loop matrix B R B^T:\n", (B @ np.diag([1.0] * 9) @ B.T).astype(int))
print("source branch current, 1 -> 4 direction:", round(i_branch[0], 6), "A  (so 4.444 A out of the + terminal)")
assert abs(i_branch[0] + 40 / 9) < 1e-12

# check against nodal analysis (node 6 as reference, source as a Norton equivalent)
G = np.zeros((6, 6)); inj = np.zeros(6)
for a, b, r, e in k33:
    p, q = int(a) - 1, int(b) - 1
    G[p, p] += 1 / r; G[q, q] += 1 / r; G[p, q] -= 1 / r; G[q, p] -= 1 / r
    inj[p] -= e / r; inj[q] += e / r               # Norton form: branch current a->b is (v_a - v_b + e)/r
vn = np.zeros(6); vn[:5] = np.linalg.solve(G[:5, :5], inj[:5])
print("nodal check, source branch current:", round((vn[0] - vn[3] - 10.0) / 1.0, 6), "A (1 -> 4)")

tree branches: ['1-4', '1-5', '1-6', '2-4', '3-4']
loop matrix B R B^T:
 [[4 2 2 1]
 [2 4 1 2]
 [2 1 4 2]
 [1 2 2 4]]
source branch current, 1 -> 4 direction: -4.444444 A  (so 4.444 A out of the + terminal)
nodal check, source branch current: -4.444444 A (1 -> 4)


The tree the search finds may differ from the one in Problem 06-05.12 (it depends on the order of the branch list), so the loop matrix may look different, but the branch currents are the same: any spanning tree gives a valid set of independent loops.

## Part 4 — A shared return

Example 06-05.5 with the motor modeled as a current sink $I_m$ (so it can be swept): the sensor (1 kΩ through 20 mΩ) and the motor return through either one shared 10 mΩ trace or separate 10 mΩ traces. The ADC reads relative to the supply's − terminal with 12 bits over 5 V.

**🔮 Predict:** how does the error scale with $I_m$ in each case?

In [5]:
LSB = 5 / 4096
def ground_offsets(Im, Rg=0.010, Rs=1000.0, r=0.020, V=5.0):
    # shared return: meshes 0 (supply, motor branch, Rg) and 1 (motor branch, sensor branch)
    shared = [("V", V, {0: 1}), ("R", r, {0: 1, 1: -1}), ("I", Im, {0: 1, 1: -1}),
              ("R", r, {1: 1}), ("R", Rs, {1: 1}), ("R", Rg, {0: 1})]
    i, _, _, _ = mesh_solve(shared, 2)
    star_sensor_current = V / (r + Rs + Rg)
    return Rg * i[0], Rg * star_sensor_current

Im = np.linspace(0, 3, 61)
off = np.array([ground_offsets(x) for x in Im])
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(Im, off[:, 0] / LSB, label="shared 10 mΩ return")
ax.plot(Im, off[:, 1] / LSB, label="star ground")
ax.set_xlabel("motor current (A)"); ax.set_ylabel("ADC error (LSB, 12-bit, 5 V)"); ax.legend(); ax.grid(True, alpha=0.3)
plt.show()
at3 = ground_offsets(3.0)
print(f"at 3 A: shared {at3[0] * 1e3:.3f} mV = {at3[0] / LSB:.1f} LSB; star {at3[1] * 1e6:.1f} uV = {at3[1] / LSB:.3f} LSB")
assert abs(at3[0] / LSB - 24.6) < 0.05

at 3 A: shared 30.050 mV = 24.6 LSB; star 50.0 uV = 0.041 LSB


With a shared return the error is a straight line in the motor current: $R_g(I_m + I_s)$, about 8.2 LSB per ampere. With a star ground it is flat and tiny, because the sensor's return carries only the sensor's own 5 mA.

## Your turn

1. In Part 1, reverse the reference direction of the shared resistor in Example 06-05.1 (swap the signs in its dictionary). Does anything change in $\mathbf{R}$ or in the answers? Why?
2. In Part 3, change the source to sit on branch 2–5 instead of 1–4. By symmetry, what source current do you expect?
3. In Part 4, the shared trace is made four times wider (2.5 mΩ). What is the error at 3 A?

<details><summary>Answers</summary>

1. Nothing changes. $\mathbf{m}_b$ flips sign, but $\mathbf{R}$ uses $\mathbf{m}_b\mathbf{m}_b^\mathsf{T}$, which is unchanged; a branch's reference direction only affects the sign of that branch's reported current.
2. The same 4.444 A: every branch of $K_{3,3}$ is equivalent to every other by symmetry (any top node can be relabeled to any other, and likewise for the bottom), so the network presents 1.25 Ω to any source branch.
3. A quarter of the offset: $2.5\ \text{mΩ}\times3.005\ \text{A} = 7.51$ mV, about 6.2 LSB. Widening helps in proportion, but only separating the returns removes the dependence on the motor current.

</details>